# Lesson 06 Tasks - Part 2: Database Design Practice (SOLUTIONS)

## Database Applications Development

**INSTRUCTOR VERSION - Complete Solutions**

## Setup: Import Libraries and Connect

In [ ]:
import sqlite3
import pandas as pd

# Connect to NBA database (for reference)
conn = sqlite3.connect('nba_5seasons.db')
print("✅ Connected to NBA database!")

---
## Task 1: Analyze NBA Database Design (10 points)

### Task 1a: Document table relationships

**SOLUTION - NBA Database ERD:**

```
┌──────────────┐
│   players    │
│──────────────│
│ PK: player_id│
│ full_name    │
└──────┬───────┘
       │
       │ 1:M
       ▼
┌────────────────────────┐
│ player_season_stats    │
│────────────────────────│
│ PK: season             │
│ PK: player_id (FK)     │
│ PK: team_id (FK)       │
│ gp, pts, reb, ast, ... │
└───────┬────────────────┘
        │
        │ M:1
        ▼
┌───────────────┐
│     teams     │
│───────────────│
│ PK: team_id   │
│ full_name     │
│ city, state   │
└───────┬───────┘
        │
        │ 1:M
        ▼
┌──────────────────────┐
│  team_game_stats     │
│──────────────────────│
│ PK: season           │
│ PK: game_id          │
│ PK: team_id (FK)     │
│ game_date, pts, ...  │
└──────────────────────┘
```

### Task 1b: Identify relationship types

**SOLUTION:**

1. **teams → team_game_stats:** 1:M (One-to-Many)
   - One team has many game stats (82+ games per season)
   - Each game stat belongs to exactly one team

2. **players → player_season_stats:** 1:M (One-to-Many)
   - One player has many season stats (one per season they played)
   - Each season stat belongs to exactly one player

3. **players ↔ teams (through player_season_stats):** M:N (Many-to-Many)
   - One player can play for multiple teams (over career or in one season if traded)
   - One team has multiple players
   - **Junction table:** player_season_stats resolves the M:N relationship

### Task 1c: Query to demonstrate M:N relationship

In [ ]:
# SOLUTION: Show LeBron James' career across multiple teams
query = """
SELECT 
    p.full_name,
    t.full_name as team_name,
    ps.season,
    ps.gp,
    ROUND(ps.pts, 1) as ppg
FROM player_season_stats ps
JOIN players p ON ps.player_id = p.player_id
JOIN teams t ON ps.team_id = t.team_id
WHERE p.full_name = 'LeBron James'
ORDER BY ps.season;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nOne player, multiple teams = M:N relationship!")

---
## Task 2: Design a School Database (15 points)

### Task 2a: Identify entities

**SOLUTION - Tables needed:**
1. **students** - Student information
2. **teachers** - Teacher information
3. **courses** - Course information
4. **enrollments** - Junction table (students ↔ courses)

### Task 2b: Design the students table

**SOLUTION:**

```sql
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE,
    grade_level INTEGER CHECK (grade_level BETWEEN 9 AND 12),
    enrollment_date TEXT NOT NULL
);
```

**Key features:**
- `student_id` as primary key (unique, never changes)
- `email` is UNIQUE (no two students same email)
- `CHECK` constraint ensures valid grade levels
- `NOT NULL` on required fields

### Task 2c: Design the courses table

**SOLUTION:**

```sql
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    teacher_id INTEGER NOT NULL,
    credits INTEGER CHECK (credits > 0),
    semester TEXT CHECK (semester IN ('Fall', 'Spring')),
    
    FOREIGN KEY (teacher_id) REFERENCES teachers(teacher_id)
);
```

**Key features:**
- Foreign key to teachers table
- CHECK constraints for valid values
- One teacher per course (simplification)

### Task 2d: Design the enrollments table

**SOLUTION:**

**Why needed:** Students ↔ Courses is a M:N relationship:
- One student takes multiple courses
- One course has multiple students
- Junction table resolves this

```sql
CREATE TABLE enrollments (
    student_id INTEGER,
    course_id INTEGER,
    enrollment_date TEXT NOT NULL,
    grade TEXT CHECK (grade IN ('A', 'B', 'C', 'D', 'F', NULL)),
    
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);
```

**Key features:**
- Composite primary key (student + course)
- Two foreign keys
- Can store additional data (enrollment date, grade)

### Task 2e: Draw school database ERD

**SOLUTION - School Database ERD:**

```
┌────────────┐
│  teachers  │
│────────────│
│ PK: teacher_id
│ full_name  │
│ department │
└─────┬──────┘
      │ 1:M
      ▼
┌────────────┐        ┌──────────────┐
│  courses   │        │   students   │
│────────────│        │──────────────│
│ PK: course_id       │ PK: student_id
│ course_name│        │ first_name   │
│ FK: teacher_id      │ last_name    │
└─────┬──────┘        └──────┬───────┘
      │                      │
      │ M:N                  │
      └──────┬───────────────┘
             ▼
      ┌─────────────┐
      │ enrollments │
      │─────────────│
      │ PK: student_id (FK)
      │ PK: course_id (FK)
      │ enrollment_date
      │ grade       │
      └─────────────┘
```

---
## Task 3: Normalize a Denormalized Table (15 points)

### Task 3a: Identify violations

**SOLUTION - Problems identified:**

1. **Violates 1NF** - Repeating groups (product1, product2, product3)
   - Can't handle more than 3 products
   - Wastes space when order has fewer products
   - Can't easily query "all orders containing product X"

2. **Data redundancy** - Customer info repeated for every order
   - If customer email changes, must update all their orders
   - Wastes storage
   - Update anomaly risk

3. **Insertion anomaly** - Can't add customer without an order

4. **No product details** - Just product names, no prices, descriptions, etc.

### Task 3b: Design normalized tables

**SOLUTION:**

```sql
-- Table 1: Customers (one row per customer)
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT
);

-- Table 2: Products (one row per product)
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    description TEXT,
    price REAL NOT NULL CHECK (price >= 0)
);

-- Table 3: Orders (one row per order)
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total REAL NOT NULL CHECK (total >= 0),
    
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- Table 4: Order Items (junction table: orders ↔ products)
CREATE TABLE order_items (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_price REAL NOT NULL,  -- Price at time of order
    
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
```

### Task 3c: Explain improvements

**SOLUTION - How normalized design solves problems:**

1. **Solves 1NF violation:**
   - No more product1, product2, product3 columns
   - Can have unlimited products per order via order_items table
   - Easy to query: `SELECT * FROM order_items WHERE product_id = X`

2. **Eliminates redundancy:**
   - Customer info stored once in customers table
   - Product info stored once in products table
   - Update customer email in one place

3. **Prevents insertion anomaly:**
   - Can add customers without orders
   - Can add products without orders

4. **Stores price history:**
   - unit_price in order_items preserves price at time of purchase
   - Product prices can change without affecting past orders

5. **Enables M:N relationship:**
   - Same product can be in multiple orders
   - Order can contain multiple products

---
## Task 4: Library Database Design (10 points)

### Task 4: Library Database Design

**SOLUTION:**

**Tables needed:**
1. patrons
2. books
3. authors
4. book_authors (junction for books ↔ authors M:N)
5. checkouts

```sql
-- Table 1: Patrons
CREATE TABLE patrons (
    patron_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE,
    phone TEXT,
    card_number TEXT UNIQUE NOT NULL,
    registration_date TEXT NOT NULL
);

-- Table 2: Authors
CREATE TABLE authors (
    author_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    birth_year INTEGER
);

-- Table 3: Books
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    isbn TEXT UNIQUE NOT NULL,
    title TEXT NOT NULL,
    publication_year INTEGER,
    copies_available INTEGER DEFAULT 1 CHECK (copies_available >= 0)
);

-- Table 4: Book_Authors (junction: books ↔ authors)
CREATE TABLE book_authors (
    book_id INTEGER,
    author_id INTEGER,
    author_order INTEGER,  -- First author, second author, etc.
    
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (author_id) REFERENCES authors(author_id)
);

-- Table 5: Checkouts
CREATE TABLE checkouts (
    checkout_id INTEGER PRIMARY KEY,
    patron_id INTEGER NOT NULL,
    book_id INTEGER NOT NULL,
    checkout_date TEXT NOT NULL,
    due_date TEXT NOT NULL,
    return_date TEXT,  -- NULL if not yet returned
    
    FOREIGN KEY (patron_id) REFERENCES patrons(patron_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id)
);
```

**ERD:**
```
┌──────────┐              ┌──────────┐
│ authors  │              │  books   │
│──────────│              │──────────│
│ PK: author_id           │ PK: book_id
│ first_name              │ title    │
│ last_name│              │ isbn     │
└────┬─────┘              └────┬─────┘
     │                         │
     │ M:N                     │
     └───────┬─────────────────┘
             ▼
      ┌──────────────┐
      │ book_authors │
      │──────────────│
      │ PK: book_id (FK)
      │ PK: author_id (FK)
      └──────────────┘
      
┌──────────┐              ┌──────────┐
│ patrons  │              │  books   │
│──────────│              │──────────│
│ PK: patron_id           │ PK: book_id
└────┬─────┘              └────┬─────┘
     │                         │
     │ 1:M                     │ 1:M
     └───────┬─────────────────┘
             ▼
      ┌──────────────┐
      │  checkouts   │
      │──────────────│
      │ PK: checkout_id
      │ FK: patron_id │
      │ FK: book_id   │
      │ checkout_date │
      │ due_date      │
      │ return_date   │
      └───────────────┘
```

**Explanation:**
- **Primary keys:** patron_id, author_id, book_id, checkout_id, composite (book_id, author_id)
- **Foreign keys:** 
  - book_authors.book_id → books.book_id
  - book_authors.author_id → authors.author_id
  - checkouts.patron_id → patrons.patron_id
  - checkouts.book_id → books.book_id
- **Junction table:** book_authors enables M:N (one book can have multiple authors, one author can write multiple books)

---
## Task 5: Reflection Questions (Bonus +5 points)

### Task 5a: When to denormalize

**SOLUTION:**

**Example:** Analytics dashboard showing daily sales summaries

**Normalized approach:**
```sql
-- Would require joining orders, customers, products tables
SELECT date, customer_name, product_name, SUM(amount)
FROM orders
JOIN customers ...
JOIN order_items ...
JOIN products ...
GROUP BY ...
```

**Denormalized approach:**
```sql
-- Create materialized view with all data pre-joined
CREATE VIEW daily_sales_summary AS
SELECT 
    date,
    customer_name,  -- Denormalized!
    product_name,   -- Denormalized!
    SUM(amount) as total
FROM ...
```

**Why denormalize:**
- Dashboard queries run constantly (thousands per day)
- JOINs across 4-5 tables are slow
- Data is read-only for dashboard
- Customer/product names rarely change
- Performance gain >> risk of stale data

**Best practice:** Keep normalized source tables, create denormalized views for reporting

### Task 5b: Composite keys in real life

**SOLUTION:**

**Example:** Movie showtime reservations

```sql
CREATE TABLE seat_reservations (
    showing_id INTEGER,  -- Which movie showing
    theater_id INTEGER,  -- Which theater
    row_number TEXT,     -- Seat row (A, B, C...)
    seat_number INTEGER, -- Seat number (1, 2, 3...)
    customer_id INTEGER,
    
    PRIMARY KEY (showing_id, theater_id, row_number, seat_number)
);
```

**Why all four columns needed:**
- `showing_id` alone: not unique (same showing has many seats)
- `theater_id` alone: not unique (theater has many showings)
- `row_number` alone: not unique (row A exists in all theaters)
- `seat_number` alone: not unique (seat 1 in every row)
- **All four together:** Uniquely identifies ONE seat in ONE showing in ONE theater

**Real-world benefit:** Prevents double-booking the same seat!

### Task 5c: Database design mistakes

**SOLUTION (Example answer):**

**Worst mistake: No primary keys**

**Why it's terrible:**
1. **Can't uniquely identify records** - How do you reference a specific customer?
2. **Can't create relationships** - Foreign keys need something to reference
3. **Can't prevent duplicates** - Same customer could be entered 10 times
4. **Can't update or delete specific records** - "Delete John Smith" might delete ALL John Smiths
5. **Can't enforce data integrity** - No way to ensure one customer has one account

**Example of disaster:**
```sql
-- Without primary key
customers: name, email, phone
John Smith, john@email.com, 555-0100
John Smith, john@email.com, 555-0100  <- Duplicate!
John Smith, j.smith@email.com, 555-0100  <- Same person?

-- Try to delete one: DELETE FROM customers WHERE name='John Smith'
-- Deletes ALL THREE! Database corrupted!
```

**Fix:**
```sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,  -- Guarantees uniqueness!
    name TEXT,
    email TEXT UNIQUE,  -- Prevent duplicate emails too
    phone TEXT
);
```

**Other terrible mistakes:**
- Storing lists in text fields (violates 1NF)
- No foreign key constraints (orphan records everywhere)
- Using application-generated IDs as keys (not database-controlled)
- God table design (everything in one massive table)

---
## Summary

**Part 2 Complete!**

Students practiced:
- ✅ Analyzing existing database designs
- ✅ Designing normalized databases from scratch
- ✅ Identifying and fixing normalization violations
- ✅ Creating ERDs
- ✅ Understanding when to normalize vs denormalize
- ✅ Applying database design principles to real-world scenarios

In [ ]:
# Close connection
conn.close()
print("✅ Connection closed. Solutions complete!")